In [0]:
#Validacion de los secrets creados, sus valores estan el portal de azure key vaul.t
usuario = dbutils.secrets.get(
    scope="electrocasa-secrets",
    key="sql-user"
)

password = dbutils.secrets.get(
    scope="electrocasa-secrets",
    key="sql-password"
)

assert usuario
assert password

print("Secretos de Azure SQL disponibles correctamente")
#print(usuario)
#print(password)

In [0]:
%sql
--CREACION DE CONEXIÓN
CREATE CONNECTION IF NOT EXISTS electrocasa_sql
TYPE SQLSERVER
OPTIONS (
  host 'analyticsdmc.database.windows.net',
  port '1433',
  user secret('electrocasa-secrets', 'sql-user'),
  password secret('electrocasa-secrets', 'sql-password')
);

In [0]:
%sql
--CREACION DE CATALOGOS FOREIGN
CREATE FOREIGN CATALOG IF NOT EXISTS electrocasa_sql_catalog
USING CONNECTION electrocasa_sql
OPTIONS (
  database 'electrocasadb'
);

In [0]:
%sql

SHOW TABLES IN electrocasa_sql_catalog.dbo;

In [0]:
%sql

DESCRIBE TABLE electrocasa_sql_catalog.dbo.trackingenvios;

In [0]:
%sql

SELECT *
FROM electrocasa_sql_catalog.dbo.trackingenvios
LIMIT 100;

In [0]:
%sql
--chequear nulos y duplicados para exploracion y ver que hacer en silver!!
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT tracking_id) AS tracking_unicos,
    SUM(CASE WHEN tracking_id IS NULL THEN 1 ELSE 0 END) AS tracking_nulos,
    SUM(CASE WHEN pedido_id IS NULL THEN 1 ELSE 0 END) AS pedido_nulos,
    SUM(CASE WHEN fecha_actualizacion IS NULL THEN 1 ELSE 0 END) AS fecha_nula
FROM electrocasa_sql_catalog.dbo.trackingenvios;

In [0]:
%sql
--DISTINCT courier
SELECT DISTINCT courier
FROM electrocasa_sql_catalog.dbo.trackingenvios
ORDER BY courier;

In [0]:
%sql
--DISTINCT estado_entrega
SELECT DISTINCT estado_entrega
FROM electrocasa_sql_catalog.dbo.trackingenvios
ORDER BY estado_entrega;

In [0]:
%sql
--tracking_id duplicados
SELECT
    tracking_id,
    COUNT(*) AS cantidad
FROM electrocasa_sql_catalog.dbo.trackingenvios
GROUP BY tracking_id
HAVING COUNT(*) > 1
ORDER BY cantidad DESC, tracking_id;

In [0]:
%sql
--muestra de duplicados
SELECT *
FROM electrocasa_sql_catalog.dbo.trackingenvios
WHERE tracking_id IN (
    'TRK000013',
    'TRK000058',
    'TRK000115',
    'TRK000204',
    'TRK000223'
)
ORDER BY tracking_id, fecha_actualizacion;

In [0]:
%sql
--distribución de estado_entrega
SELECT
    estado_entrega,
    COUNT(*) AS cantidad
FROM electrocasa_sql_catalog.dbo.trackingenvios
GROUP BY estado_entrega
ORDER BY cantidad DESC;

In [0]:
%sql

SELECT COUNT(*) AS total_registros
FROM electrocasa_sql_catalog.dbo.trackingenvios;

In [0]:
%sql

CREATE OR REPLACE TABLE electrocasa_dev.bronze.tracking_envios AS
SELECT
    *,
    current_timestamp() AS fec_ingesta,
    'azure_sql' AS sistema_origen,
    concat(
        'tracking_',
        date_format(current_timestamp(), 'yyyyMMdd_HHmmss')
    ) AS id_lote
FROM electrocasa_sql_catalog.dbo.trackingenvios;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT tracking_id) AS tracking_unicos,
    COUNT(DISTINCT sistema_origen) AS sistemas_origen,
    COUNT(DISTINCT id_lote) AS lotes
FROM electrocasa_dev.bronze.tracking_envios;

In [0]:
%sql

SELECT
    tracking_id,
    COUNT(*) AS registros,
    COUNT(DISTINCT concat_ws(
        '||',
        pedido_id,
        courier,
        estado_entrega,
        sucursal_origen,
        CAST(fecha_actualizacion AS STRING)
    )) AS versiones_distintas
FROM electrocasa_dev.bronze.tracking_envios
GROUP BY tracking_id
HAVING COUNT(*) > 1
ORDER BY tracking_id;

In [0]:
 %sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT tracking_id) AS tracking_unicos,
    COUNT(DISTINCT estado_entrega) AS estados_distintos,

    SUM(CASE
        WHEN fecha_actualizacion IS NULL
        THEN 1 ELSE 0
    END) AS fechas_nulas

FROM electrocasa_dev.silver.tracking_envios;

In [0]:
%sql

SELECT
    COUNT(*) AS filas_nulas_bronze,
    COUNT(DISTINCT tracking_id) AS tracking_nulos_unicos
FROM electrocasa_dev.bronze.tracking_envios
WHERE fecha_actualizacion IS NULL;